## Tool binding

Tool binding is the step where you register tools with a language model(LLM) so that:

1. The LLM knows what tools are available.
2. It knows what each tool are available.
3. It knows what input foramt to use.

In [30]:
!pip install requests

In [31]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import tool
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage


In [32]:
#tool create

@tool
def multiply(a:int,b:int) -> int:
    """Given 2 numbers a and b this tool returns their product"""
    return a * b

In [33]:
print(multiply.invoke({'a':3 , 'b':11}))

33


In [34]:
multiply.name

'multiply'

In [35]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [36]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [37]:
load_dotenv()

True

In [38]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature = 0
    )

In [39]:
# tool binding

llm_with_tools = llm.bind_tools([multiply])

## Tool calling

Tool calling is the process where the LLM decides during a conversation or task , that it needs to use a specific tool (function) --- and generate a structured output with :

1. name of the tool
2. and the arguements to call it with



The LLM does not actually run the tool , it just suggests the tool and the input arguements . The actual execution is handled by langchain or us.

## Tool Execution

is the step where the actual python function (tool) is run using the input arguements that the LLM suggested during tool calling.

In [40]:
result = llm_with_tools.invoke('can you multiply 3 with 20')

In [41]:
result.tool_calls[0]

{'name': 'multiply',
 'args': {'b': 20, 'a': 3},
 'id': '1f1996b3-e348-4132-a58e-a97395676c73',
 'type': 'tool_call'}

In [42]:
multiply.invoke(result.tool_calls[0])

ToolMessage(content='60', name='multiply', tool_call_id='1f1996b3-e348-4132-a58e-a97395676c73')

In [43]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ce354-92f2-7272-bed6-5e8d5731da37-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [44]:
query = HumanMessage('can you multiply 4 with 69')

In [45]:
messages = [query]

In [46]:
messages

[HumanMessage(content='can you multiply 4 with 69', additional_kwargs={}, response_metadata={})]

In [47]:
result = llm_with_tools.invoke(messages)

In [48]:
messages.append(result)

In [49]:
messages

[HumanMessage(content='can you multiply 4 with 69', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 4, "b": 69}'}, '__gemini_function_call_thought_signatures__': {'10de14b9-4c96-4a92-8454-80dfbf9760ed': 'CvUBAb4+9vt3Hh9/APwxaUuj4vxIBe8oEvLzxZOhgrLEa1fF/IiFWa3u6CQLk/clySj3ArlUylajScygBbTkbcYghDIvCdyng+OMyxWHBg1A221a3BxR5fapFs+a9XepkxVN525mL07/q/ZM6zSMLzWkuFeOUamndj+HU3ayY6BiR80rwnvgIbTdMxoUCM09qqJOVzsw6Wr4USBfjxJGRedQ2k6oSyQRgV4bo6niYn1e5aSZAT9U/5W/WAqF/XhV4sj/NZVZJbk6WW/6oEAMkNLBfDCqgZshW/fK3qaQ25VbUnoui7B477WuzWeARafjBo0ETZa22Lo='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ce354-9a89-7f83-9ff6-cbcf2a3d3d00-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 69}, 'id': '10de14b9-4c96-4a92-8454-80dfbf9760ed', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [50]:
tool_result = multiply.invoke(result.tool_calls[0])

In [51]:
messages.append(tool_result)

In [52]:
messages

[HumanMessage(content='can you multiply 4 with 69', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 4, "b": 69}'}, '__gemini_function_call_thought_signatures__': {'10de14b9-4c96-4a92-8454-80dfbf9760ed': 'CvUBAb4+9vt3Hh9/APwxaUuj4vxIBe8oEvLzxZOhgrLEa1fF/IiFWa3u6CQLk/clySj3ArlUylajScygBbTkbcYghDIvCdyng+OMyxWHBg1A221a3BxR5fapFs+a9XepkxVN525mL07/q/ZM6zSMLzWkuFeOUamndj+HU3ayY6BiR80rwnvgIbTdMxoUCM09qqJOVzsw6Wr4USBfjxJGRedQ2k6oSyQRgV4bo6niYn1e5aSZAT9U/5W/WAqF/XhV4sj/NZVZJbk6WW/6oEAMkNLBfDCqgZshW/fK3qaQ25VbUnoui7B477WuzWeARafjBo0ETZa22Lo='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019ce354-9a89-7f83-9ff6-cbcf2a3d3d00-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 69}, 'id': '10de14b9-4c96-4a92-8454-80dfbf9760ed', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input

In [53]:
llm_with_tools.invoke(messages).content

'The product of 4 and 69 is 276.'